In [1]:
import pandas as pd
import polars as pl

In [2]:
data = {
    "column1": [1, 2, 3],
    "column2": ["a", "b", "c"]
}

df = pl.DataFrame(data)
print(df)


shape: (3, 2)
┌─────────┬─────────┐
│ column1 ┆ column2 │
│ ---     ┆ ---     │
│ i64     ┆ str     │
╞═════════╪═════════╡
│ 1       ┆ a       │
│ 2       ┆ b       │
│ 3       ┆ c       │
└─────────┴─────────┘


In [4]:
pl.read_csv("1.csv")

1,1_duplicated_0
i64,i64
2,2
3,3
4,4
5,5
6,6
7,7


In [5]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "score": [85.5, 90.0, 92.5]
})

# Save it as a Parquet file
df.write_parquet("file.parquet")



In [6]:
pl.read_parquet("file.parquet")

id,name,score
i64,str,f64
1,"""Alice""",85.5
2,"""Bob""",90.0
3,"""Charlie""",92.5


In [8]:
df.select("name")          # Select column(s)
df.filter(pl.col("score") > 90)     # Filter rows

id,name,score
i64,str,f64
3,"""Charlie""",92.5


In [12]:
df1 = pl.DataFrame({
    "category": ["A", "B", "A", "B", "C"],
    "sales": [100, 200, 150, 300, 400],
    "price": [10, 20, 15, 30, 25],
    "quantity": [5, 3, 2, 4, 6],
    "date": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"],
    "id": [1, 2, 3, 4, 5],
    "score": [95, 85, 90, 92, 88],
    "name": ["Alice", "Bob", "Charlie", "David", "Eva"]
})

In [27]:
df1 = pl.DataFrame({
    "category": ["A", "B", "A", "B", "C"],
    "sales": [100, 200, 150, 300, 400],
    "price": [10, 20, 15, 30, 25],
    "quantity": [5, 3, 2, 4, 6],
    "date": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"],
    "id": [1, 2, 3, 4, 5],
    "score": [95, 85, 90, 92, 88],
    "name": ["Alice", "Bob", "Charlie", "David", "Eva"]
})

result = df1.group_by("category").agg([
    pl.col("sales").sum().alias("total_sales")
])

print(result)

shape: (3, 2)
┌──────────┬─────────────┐
│ category ┆ total_sales │
│ ---      ┆ ---         │
│ str      ┆ i64         │
╞══════════╪═════════════╡
│ B        ┆ 500         │
│ C        ┆ 400         │
│ A        ┆ 250         │
└──────────┴─────────────┘


In [14]:
df1.select([
    (pl.col("price") * pl.col("quantity")).alias("total"),
    pl.col("category").alias("group")
])

total,group
i64,str
50,"""A"""
60,"""B"""
30,"""A"""
120,"""B"""
150,"""C"""


In [20]:
df1.sort("date", descending=True)
df1

category,sales,price,quantity,date,id,score,name
str,i64,i64,i64,str,i64,i64,str
"""A""",100,10,5,"""2024-01-01""",1,95,"""Alice"""
"""B""",200,20,3,"""2024-01-02""",2,85,"""Bob"""
"""A""",150,15,2,"""2024-01-03""",3,90,"""Charlie"""
"""B""",300,30,4,"""2024-01-04""",4,92,"""David"""
"""C""",400,25,6,"""2024-01-05""",5,88,"""Eva"""


In [24]:
other_df = pl.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "region": ["East", "West", "North", "South", "Central"]
})


df1.join(other_df, on="id", how="inner")


category,sales,price,quantity,date,id,score,name,region
str,i64,i64,i64,str,i64,i64,str,str
"""A""",100,10,5,"""2024-01-01""",1,95,"""Alice""","""East"""
"""B""",200,20,3,"""2024-01-02""",2,85,"""Bob""","""West"""
"""A""",150,15,2,"""2024-01-03""",3,90,"""Charlie""","""North"""
"""B""",300,30,4,"""2024-01-04""",4,92,"""David""","""South"""
"""C""",400,25,6,"""2024-01-05""",5,88,"""Eva""","""Central"""


In [19]:
ldf = df1.lazy()
result = ldf.filter(pl.col("score") > 90).select("name").collect()
result

name
str
"""Alice"""
"""David"""
